In [ ]:
import polars as pl
from pathlib import Path


def direction(lo, hi):
    """Classify a 95% CI on a fold/ratio scale relative to the null (1.0)."""
    if lo is None or hi is None:
        return None
    if lo > 1.0:
        return "increased"
    if hi < 1.0:
        return "decreased"
    return "n.s."

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]
label_map = {
    "fungi_mit": "Fungi (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_plt": "Green algae (plastid)",
    "plants_plt": "Plants (plastid)",
    "protists_plt": "Protists (plastid)",
}

In [ ]:
rows_s3 = []

for g in groups:
    gdir = BASE / g

    icc_row = pl.read_csv(
        gdir / "brms_type_length" / "brms_sigma_icc_row.tsv", separator="\t"
    ).row(0, named=True)

    rows_s3.append(
        {
            "group": label_map[g],
            "igr_regions": icc_row["N_regions"],
            "N_taxa": icc_row["N_taxa"],
            "sigma_ratio_conv": icc_row["sigma_ratio_conv"],
            "sigma_ratio_conv_lo": icc_row["sigma_ratio_conv_lo"],
            "sigma_ratio_conv_hi": icc_row["sigma_ratio_conv_hi"],
            "conv_dispersion": direction(
                icc_row["sigma_ratio_conv_lo"], icc_row["sigma_ratio_conv_hi"]
            ),
            "sigma_ratio_div": icc_row["sigma_ratio_div"],
            "sigma_ratio_div_lo": icc_row["sigma_ratio_div_lo"],
            "sigma_ratio_div_hi": icc_row["sigma_ratio_div_hi"],
            "div_dispersion": direction(
                icc_row["sigma_ratio_div_lo"], icc_row["sigma_ratio_div_hi"]
            ),
            "sigma_baseline_sd": icc_row["sigma_baseline_sd"],
            "icc_len": icc_row["icc_len_median"],
            "icc_len_lo": icc_row["icc_len_lo"],
            "icc_len_hi": icc_row["icc_len_hi"],
            "icc_type": icc_row["icc_type_median"],
            "icc_type_lo": icc_row["icc_type_lo"],
            "icc_type_hi": icc_row["icc_type_hi"],
            "icc_both": icc_row["icc_both_median"],
            "icc_both_lo": icc_row["icc_both_lo"],
            "icc_both_hi": icc_row["icc_both_hi"],
        }
    )

s3 = pl.DataFrame(rows_s3)

float_cols = [col for col in s3.columns if s3[col].dtype in [pl.Float32, pl.Float64]]
s3 = s3.with_columns([pl.col(col).round(3) for col in float_cols])

s3.write_csv(BASE / "code" / "supplementary_table3.tsv", separator="\t")